# Silver Incremental — Order Items (SCD1 via CDF + MERGE)
**GlobalMart | Tredence DE Advanced Training | Day 12 Pattern**

| | |
|---|---|
| **Source** | `gbmart.bronze.order_items` — CDF enabled (Lakeflow Connect CDC) |
| **Target** | `gbmart.silver.order_items` |
| **SCD Type** | SCD1 — line items are immutable transactional facts; new lines are inserts only |

### Dependency
Run `12_orders_incremental_scd1_merge.ipynb` **before** this notebook.
`OR-900001` and `OR-900002` must exist in `silver.orders` before order_items for
those orders can pass referential integrity.

### The flow
| Step | What it does |
|---|---|
| 1 | Baseline row count |
| 2 | Inspect Bronze history — find the version to read from |
| 3 | CDF read — only the new/changed order_item rows |
| 4 | Referential integrity check (new items must have a parent order + product in Silver) |
| 5 | SCD1 MERGE into silver.order_items |
| 6 | Verify |

## Step 1 — Setup + Baseline

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BRONZE_TABLE = "harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.order_items"
SILVER_TABLE = "harsh_kumar01_npmentorskool_onmicrosoft_com.silver.order_items"

baseline_count = spark.table(SILVER_TABLE).count()
print(f"silver.order_items before this run : {baseline_count:,}")

## Step 2 — Inspect Bronze History
Find the version the initial full-load notebook already consumed.
CDF reads everything strictly **after** `LAST_PROCESSED_VERSION`.

In [ ]:
# DESCRIBE HISTORY on bronze.orders/order_items fails here with
# STREAMING_TABLE_OPERATION_NOT_ALLOWED.REQUIRES_SHARED_COMPUTE -- these Bronze
# tables are Streaming Tables (Lakeflow-managed), and DESCRIBE HISTORY on a
# Streaming Table requires a Shared cluster or a SQL warehouse, not the
# Assigned/No-Isolation cluster this notebook runs on. table_changes() is not
# subject to that restriction (cells below already use it successfully), so we
# rebuild the same per-version summary from it instead of DESCRIBE HISTORY.
# If you switch this notebook to a Shared cluster or SQL warehouse, the original
# spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}")...display() also works fine.
(
    spark.sql(f"""
        SELECT _commit_version, _commit_timestamp, _change_type, COUNT(*) AS row_count
        FROM table_changes('{BRONZE_TABLE}', 0)
        GROUP BY _commit_version, _commit_timestamp, _change_type
        ORDER BY _commit_version, _change_type
    """)
    .display()
)

In [ ]:
%sql
SELECT *
FROM table_changes(
  'harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.order_items',
  0,
  10
);

In [ ]:
%sql
-- Version 1
SELECT *
FROM table_changes(
  'harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.order_items',
  1
);

In [ ]:
%sql
-- Version 2
SELECT *
FROM table_changes(
  'harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.order_items',
  2
);

In [ ]:
%sql
-- Version 3
SELECT *
FROM table_changes(
  'harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.order_items',
  3
);

In [ ]:
%sql
-- Version 4
SELECT *
FROM table_changes(
  'harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.order_items',
  3
);

In [ ]:
LAST_PROCESSED_VERSION = 3   # <-- update from history output above

## Step 3 — CDF Read: Only the Changes
For the Day 12 scenario: if `05_make_order_changes_for_incremental.sql` included the
optional `INSERT INTO order_items` block, new line items for `OR-900001`/`OR-900002`
land here via Lakeflow. If not, this CDF read returns 0 rows — that's correct.

In [ ]:
cdf_df_raw = (
    spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", LAST_PROCESSED_VERSION + 1)
        .table(BRONZE_TABLE)
        .filter(col("_change_type").isin(["insert", "update_postimage"]))
)

# Defensive: keep only the latest event per order_item_id, ranked by
# _commit_version (Delta's own monotonic commit counter). Line items are
# insert-only in the current Day 12 scenario so this is normally a no-op, but
# it keeps this notebook safe from the same
# DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE failure fixed in
# notebook 12, if a line item is ever corrected within one incremental batch.
item_window = Window.partitionBy("orderitemid").orderBy(col("_commit_version").desc())
cdf_df = (
    cdf_df_raw
    .withColumn("_rn", row_number().over(item_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

changed_count = cdf_df.count()
raw_count = cdf_df_raw.count()
print(f"Changed rows via CDF (insert/update_postimage only) : {raw_count:,}")
print(f"Changed rows after de-duplicating to latest per order_item_id : {changed_count:,}")

if changed_count > 0:
    cdf_df.select("orderitemid", "orderid", "productid", "quantity", "_change_type", "_commit_version").display()
else:
    print("No new order_items in this batch — nothing to merge.")

## Step 4 — Referential Integrity Check
New order_items must have a parent order in `silver.orders` and a valid product in
`silver.products`. If `notebook 12` ran first, `OR-900001`/`OR-900002` should already
be there.

In [ ]:
if changed_count > 0:
    silver_orders   = spark.table("harsh_kumar01_npmentorskool_onmicrosoft_com.silver.orders").select("order_id")
    silver_products = spark.table("harsh_kumar01_npmentorskool_onmicrosoft_com.silver.products").select("product_id").distinct()

    orphan_orders   = cdf_df.join(silver_orders,   cdf_df.OrderID   == silver_orders.order_id,   "left_anti")
    orphan_products = cdf_df.join(silver_products, cdf_df.ProductID == silver_products.product_id, "left_anti")

    print(f"order_items with no matching silver.order   : {orphan_orders.count():,}  (expected 0)")
    print(f"order_items with no matching silver.product : {orphan_products.count():,}  (expected 0)")

    if orphan_orders.count() > 0:
        print("\n[WARNING] Orphaned orders — run notebook 12 first, then re-run this notebook")
        orphan_orders.select("orderitemid", "orderid").display()
else:
    print("Skipped — no rows to check.")

## Step 5 — Transform + SCD1 MERGE into silver.order_items
Same minimal transform as the full-load notebook — rename to snake_case, add audit
column. Line-item amounts are not computed here; `fact_sales` derives them at Gold
by joining to `silver.products`.

In [ ]:
if changed_count > 0:
    silver_df = cdf_df \
        .withColumnRenamed("orderitemid", "order_item_id") \
        .withColumnRenamed("orderid",     "order_id") \
        .withColumnRenamed("productid",   "product_id") \
        .withColumn("_silver_updated_at", current_timestamp()) \
        .select("order_item_id", "order_id", "product_id", "quantity", "updated_at", "_silver_updated_at")

    silver_table = DeltaTable.forName(spark, SILVER_TABLE)

    (silver_table.alias("tgt")
        .merge(silver_df.alias("src"), "tgt.order_item_id = src.order_item_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"MERGE complete")
    print(f"silver.order_items after this run: {spark.table(SILVER_TABLE).count():,}")
else:
    print("No rows to merge — silver.order_items unchanged.")

In [ ]:
if changed_count > 0:
    # Validation — duplicate order_item_ids. Should always be 0 after the Step 3
    # de-duplication above; stop here rather than risk a bad MERGE if it isn't.
    dupes = silver_df.groupBy("order_item_id").count().filter("count > 1")
    dupe_count = dupes.count()
    print(f"Duplicate order_item_ids in silver_df : {dupe_count}  (expected 0)")
    if dupe_count > 0:
        dupes.display()
        raise AssertionError(f"Found {dupe_count} duplicate order_item_id(s) -- fix upstream de-duplication before re-running.")

    # Validation — merge metrics, pulled directly from Delta's own transaction log
    merge_metrics_row = (
        spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}")
        .orderBy(desc("version"))
        .limit(1)
        .select("version", "operation", "operationMetrics")
        .collect()[0]
    )
    print(f"\nMERGE metrics (silver.order_items Delta version {merge_metrics_row['version']}):")
    for k, v in merge_metrics_row["operationMetrics"].items():
        if k.startswith("numTarget") or k.startswith("numSource"):
            print(f"  {k:35s}: {v}")
else:
    print("Skipped duplicate check + merge metrics — no rows were merged this run.")

## Step 6 — Verify

In [ ]:
df = spark.table(SILVER_TABLE)
print(f"silver.order_items row count : {df.count():,}  (was {baseline_count:,} before run)")

# Check if new order_items for OR-900001 / OR-900002 landed
new_items = df.filter(col("order_id").isin(["OR-900001", "OR-900002"]))
print(f"\norder_items for OR-900001/OR-900002 : {new_items.count():,} rows")
if new_items.count() > 0:
    new_items.select("order_item_id", "order_id", "product_id", "quantity").display()

### Validation — order_items Can Join to silver.orders
The referential-integrity check in Step 4 only looked at *this batch*. This
checks the *whole* `silver.order_items` table against the current `silver.orders`
-- the same join Gold `fact_sales` will do. Any row that fails here is a row
Gold will silently drop.

In [ ]:
orders_for_join = (
    spark.table("harsh_kumar01_npmentorskool_onmicrosoft_com.silver.orders")
    .select(col("order_id").alias("_join_order_id"))
)
unjoinable = df.join(orders_for_join, df.order_id == orders_for_join._join_order_id, "left_anti")
unjoinable_count = unjoinable.count()

print(f"order_items with no matching silver.orders row : {unjoinable_count:,}  (expected 0)")
if unjoinable_count > 0:
    print("[WARNING] These order_items cannot join to silver.orders -- Gold fact_sales will drop them.")
    print("If notebook 12 hasn't been re-run with the Step 3 fix yet, run it first, then re-run this notebook.")
    unjoinable.select("order_item_id", "order_id").display()

## Reset (if needed)

In [ ]:
# spark.sql("DELETE FROM gbmart.silver.order_items WHERE order_id IN ('OR-900001','OR-900002')")
# print("New order_items removed")